In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
# Import the point-level extraction and outlier functions
import pandas as pd
from ML_LC_Classifier import (
    detect_outliers_per_class,
    load_and_extract_points,
    remove_outliers,
    save_cleaned_points,
)


raster_path = r"C:\AFLICM_Test\Imagery\reswir\*.tif"
final_s2 = stack_rasters(raster_path, output_path=project_root / "input_data" / "feature_stack.tif",
                         target_crs = "EPSG:32751",
                         resolution=20,
                         band_names = ["B2", "B3", "B4", "B5", "B6", "B8", "B11", "B12"])

In [4]:
# Point-level outlier detection using two rasters sampled independently
RASTER_PATH_1 = r"C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif"
RASTER_PATH_2 = r"C:\AFLICM_Test\Imagery\reswir\reswir_mosaic_TL_sentinel2.tif"
POINTS_PATH = r"C:\Users\AFahrezi\Documents\GitHub\Improve_pixel_based_lc_classification\input_data\samples_TL.geojson"
CLASS_FIELD = "ID"
CLASS_NAME_FIELD = "LULC_Type"
ID_FIELD = "point_id"
REMOVE_OUTLIERS = True

metadata_cols = [ID_FIELD, CLASS_FIELD, CLASS_NAME_FIELD]

features_1 = load_and_extract_points(
    raster_path=RASTER_PATH_1,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
)
features_2 = load_and_extract_points(
    raster_path=RASTER_PATH_2,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
)

features_1 = features_1.rename(
    columns={
        column: f"raster_1_{column}"
        for column in features_1.columns
        if column not in metadata_cols
    }
)
features_2 = features_2.rename(
    columns={
        column: f"raster_2_{column}"
        for column in features_2.columns
        if column not in metadata_cols
    }
)

point_features = features_1.merge(
    features_2,
    on=metadata_cols,
    how="inner",
    validate="one_to_one",
)

feature_cols = [
    column for column in point_features.columns
    if column not in metadata_cols
]
flags = detect_outliers_per_class(
    df=point_features,
    feature_cols=feature_cols,
    class_col=CLASS_FIELD,
    id_col=ID_FIELD,
    class_name_col=CLASS_NAME_FIELD,
)

flags.to_csv(
    project_root / "output" / "training_point_outlier_flags_samples_TL.csv",
    index=False,
)

# Review flags before using the cleaned table for model training.
cleaned_point_features = (
    remove_outliers(point_features, flags, id_col=ID_FIELD)
    if REMOVE_OUTLIERS
    else point_features.copy()
)
cleaned_point_features.to_csv(
    project_root / "output" / "training_points_cleaned_samples_TL.csv",
    index=False,
)

if REMOVE_OUTLIERS:
    save_cleaned_points(
        points_path=POINTS_PATH,
        flags=flags,
        output_path=project_root / "output" / "training_points_cleaned_samples_TL.shp",
        id_col=ID_FIELD,
    )

print(f"Raster 1 samples: {len(features_1)}")
print(f"Raster 2 samples: {len(features_2)}")
print(f"Shared samples used: {len(point_features)}")
print(f"Flagged outliers: {flags['outlier'].sum()}")
print(f"Samples after filtering: {len(cleaned_point_features)}")
flags[flags["outlier"]]


Raster 1 samples: 2636
Raster 2 samples: 2636
Shared samples used: 2636
Flagged outliers: 138
Samples after filtering: 2498


,point_id,ID,LULC_Type,outlier,score,n_in_class
2353,4,15,Waterbody,True,0.814459,287
1744,2137,13,Cleared land,True,0.814275,267
2312,1259,14,Built-up area,True,0.779241,518
649,6,7,Tree based monoculture,True,0.770403,103
2352,3,15,Waterbody,True,0.766863,287
...,...,...,...,...,...,...
2204,909,14,Built-up area,True,0.560168,518
139,1044,1,Undisturbed dryland forest,True,0.559998,216
1018,401,10,Grassland,True,0.558717,287
947,330,10,Grassland,True,0.558376,287
